# ML & Sensing Final Project: Breathing Pattern Detection

**Authors**: 

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

from scipy.fft import fft, fftfreq

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
# Load dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arashnic/lung-dataset")

print("Path to dataset files:", path)

100%|██████████| 65.5M/65.5M [00:02<00:00, 33.5MB/s]

Extracting files...


Path to dataset files: /Users/maanvisarwadi/.cache/kagglehub/datasets/arashnic/lung-dataset/versions/1


In [ ]:
def extract_audio_features(file_path):
    audio, sr = librosa.load(file_path, sr=22050)

    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    zero_crossing_rate = librosa.feature.zero_crossing_rate(audio)

    features = np.hstack([
        np.mean(mfccs, axis=1),
        np.mean(spectral_centroid),
        np.mean(zero_crossing_rate)
    ])

    return features

In [ ]:
# Column names for PhyPhox accelerometer CSV
# Adjust if your app exports different column names
ACCEL_COLS = {
    'time': 'time (s)',
    'x':    'Linear Acceleration x (m/s^2)',
    'y':    'Linear Acceleration y (m/s^2)',
    'z':    'Linear Acceleration z (m/s^2)',
}

LABELS = {'resting': 0, 'active': 1}
CLASS_NAMES = ['Resting', 'Active']

DATA_ROOT   = 'data'          # root folder (see directory structure above)
AUDIO_SR    = 22050           # audio sample rate (Hz)
N_MFCC      = 13              # number of MFCC coefficients
ACCEL_SR    = 100             # accelerometer sample rate (Hz); PhyPhox default
BREATH_LOW  = 0.1             # breathing band lower bound (Hz) — ~6 breaths/min
BREATH_HIGH = 0.5             # breathing band upper bound (Hz) — ~30 breaths/min

# Matching feature names (used for importance plots later)
AUDIO_FEATURE_NAMES = (
    [f'mfcc_{i}_mean' for i in range(N_MFCC)] +
    [f'mfcc_{i}_std'  for i in range(N_MFCC)] +
    ['centroid_mean', 'centroid_std',
     'zcr_mean',      'zcr_std',
     'rms_mean',      'rms_std',
     'rolloff_mean',  'rolloff_std']
)

print(f"Audio feature vector length: {len(AUDIO_FEATURE_NAMES)}")


In [ ]:
def load_accel_csv(file_path):
    """
    Loads a PhyPhox accelerometer CSV and returns a DataFrame
    with standardized columns: x, y, z (in m/s^2).
    """
    df = pd.read_csv(file_path)

    # Rename columns to x, y, z regardless of PhyPhox's verbose names
    rename_map = {
        ACCEL_COLS['x']: 'x',
        ACCEL_COLS['y']: 'y',
        ACCEL_COLS['z']: 'z',
    }
    df = df.rename(columns=rename_map)

    # Fallback: if columns are just named x, y, z already
    for col in ['x', 'y', 'z']:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in {file_path}. "
                             f"Available columns: {list(df.columns)}")

    return df[['x', 'y', 'z']].dropna()


def extract_accel_features(file_path, accel_sr=ACCEL_SR,
                           breath_low=BREATH_LOW, breath_high=BREATH_HIGH):
    """
    Returns a 1D feature vector from an accelerometer CSV.
    Total features: 15
    """
    df = load_accel_csv(file_path)

    # ── Time-domain features ─────────────────────────────────────────────────
    magnitude = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2).values

    time_features = np.array([
        np.mean(magnitude),
        np.std(magnitude),
        np.max(magnitude),
        np.min(magnitude),
        np.max(magnitude) - np.min(magnitude),  # range
        np.var(magnitude),
        np.mean(df['x'].values), np.std(df['x'].values),
        np.mean(df['y'].values), np.std(df['y'].values),
        np.mean(df['z'].values), np.std(df['z'].values),
    ])

    # ── Frequency-domain features (FFT) ──────────────────────────────────────
    n       = len(magnitude)
    freqs   = fftfreq(n, d=1.0 / accel_sr)
    fft_mag = np.abs(fft(magnitude))

    # Keep only positive frequencies
    pos_mask   = freqs > 0
    freqs_pos  = freqs[pos_mask]
    fft_pos    = fft_mag[pos_mask]

    # Breathing band (0.1 – 0.5 Hz)
    breath_mask    = (freqs_pos >= breath_low) & (freqs_pos <= breath_high)
    breath_power   = np.sum(fft_pos[breath_mask]) if breath_mask.sum() > 0 else 0.0
    dominant_freq  = (
        freqs_pos[breath_mask][np.argmax(fft_pos[breath_mask])]
        if breath_mask.sum() > 0 else 0.0
    )
    total_power    = np.sum(fft_pos) + 1e-8
    breath_ratio   = breath_power / total_power

    freq_features = np.array([dominant_freq, breath_power, breath_ratio])

    return np.hstack([time_features, freq_features])


ACCEL_FEATURE_NAMES = [
    'mag_mean', 'mag_std', 'mag_max', 'mag_min', 'mag_range', 'mag_var',
    'x_mean', 'x_std', 'y_mean', 'y_std', 'z_mean', 'z_std',
    'dominant_freq_hz', 'breath_band_power', 'breath_power_ratio',
]

print(f"Accelerometer feature vector length: {len(ACCEL_FEATURE_NAMES)}")

In [ ]:
ALL_FEATURE_NAMES = AUDIO_FEATURE_NAMES + ACCEL_FEATURE_NAMES
print(f"Total fused feature vector length: {len(ALL_FEATURE_NAMES)}")


def build_dataset(data_root=DATA_ROOT):
    """
    Walks data_root/<label>/audio/ and data_root/<label>/accel/,
    pairs files by sorted name, extracts features, and returns X, y.
    """
    X_rows, y_rows = [], []

    for label_name, label_val in LABELS.items():
        audio_dir = os.path.join(data_root, label_name, 'audio')
        accel_dir = os.path.join(data_root, label_name, 'accel')

        if not os.path.isdir(audio_dir) or not os.path.isdir(accel_dir):
            print(f"[WARNING] Missing directory for label '{label_name}'. Skipping.")
            continue

        audio_files = sorted(f for f in os.listdir(audio_dir) if f.endswith('.wav'))
        accel_files = sorted(f for f in os.listdir(accel_dir) if f.endswith('.csv'))

        if len(audio_files) != len(accel_files):
            print(f"[WARNING] '{label_name}': {len(audio_files)} audio files "
                  f"but {len(accel_files)} accel files — using min of both.")

        pairs = list(zip(audio_files, accel_files))
        print(f"[{label_name.upper()}] {len(pairs)} paired sessions found.")

        for audio_fname, accel_fname in pairs:
            try:
                audio_feat = extract_audio_features(
                    os.path.join(audio_dir, audio_fname)
                )
                accel_feat = extract_accel_features(
                    os.path.join(accel_dir, accel_fname)
                )
                fused = np.concatenate([audio_feat, accel_feat])
                X_rows.append(fused)
                y_rows.append(label_val)

            except Exception as e:
                print(f"  [ERROR] {audio_fname} / {accel_fname}: {e}")

    X = np.array(X_rows)
    y = np.array(y_rows)
    return X, y


X, y = build_dataset()

print(f"\nDataset shape : X = {X.shape}, y = {y.shape}")
print(f"Class balance : Resting = {np.sum(y == 0)}, Active = {np.sum(y == 1)}")

In [ ]:
# Visualize accelerometer signal
plt.figure(figsize=(12, 4))
plt.plot(accel_df['x'], label='X-axis')
plt.plot(accel_df['y'], label='Y-axis')
plt.plot(accel_df['z'], label='Z-axis')
plt.title("Accelerometer Signal")
plt.xlabel("Time")
plt.ylabel("Acceleration")
plt.legend()
plt.show()

In [ ]:
# Extract features from accelerometer data
def extract_accelerometer_features(df):

    magnitude = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

    features = [
        np.mean(magnitude),
        np.std(magnitude),
        np.max(magnitude),
        np.min(magnitude),
        np.var(magnitude)
    ]

    return np.array(features)